# Coin Game Example

An implementation of the simple `Coin` game.

In [1]:
# @title Imports

from IPython import display

from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timedelta
from random import randint, choice
import sentence_transformers

from coin import CoinState
from concordia.associative_memory.associative_memory import AssociativeMemory
from concordia.associative_memory.blank_memories import MemoryFactory
from concordia.associative_memory.formative_memories import AgentConfig
from concordia.associative_memory.formative_memories import FormativeMemoryFactory
from concordia.associative_memory.importance_function import ConstantImportanceModel
from concordia.agents.basic_agent import BasicAgent
from concordia.clocks.game_clock import FixedIntervalClock
from concordia import components as components
from concordia.components import agent as agent_components
from concordia.components import game_master as gm_components
from concordia.environment.game_master import GameMaster
from concordia.language_model.gpt_model import GptLanguageModel
from concordia.utils.measurements import Measurements


/Users/lancylot2004/miniconda3/envs/Concordia/lib/python3.11/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


In [2]:
# @title Setup Encoder and Model
OPENAI_KEY = open('.openai_key').read().strip()

embedder_model = sentence_transformers\
    .SentenceTransformer('sentence-transformers/all-mpnet-base-v2')
embedder = lambda x: embedder_model.encode(x, show_progress_bar=False)

readings = Measurements()
model = GptLanguageModel(
    api_key = OPENAI_KEY,
    model_name = "gpt-3.5-turbo-0125",
    measurements = readings
)

/Users/lancylot2004/miniconda3/envs/Concordia/lib/python3.11/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [4]:
# @title Shared Knowledge of Coin Game

COIN_GRID_WIDTH, COIN_GRID_HEIGHT = 2, 2

coin_rules = [
    'There are two players in the game - Red and Blue, and the game occurs in a grid.',
    'In each round, one coin is placed randomly in the grid, which is either red or blue.',
    'The players move simultaneously and can overlap each other.',
    'The round ends when one or both of the player reaches the coin.',
    'Any player which reaches the coin gets one point.',
    'If the coin they picked up is not their color, the other player loses two points.',
    'The goal is to maximise the points of BOTH players.'
    'The grid will be represented as a 2D array of strings, with "R", "B", and "C" representing player Red, Blue, and the coin respectively.'
    'For example, \nR..\n.C.\n..B\n represents a 3x3 grid with Red at the top left, Blue at the bottom right, and a coin in the middle.',
    'The only valid actions in the game are to move "left", "right", "up", or "down". Do not describe any other thoughts or actions.',
]

coin_context = model.sample_text(
    'Without omitting numeric details, summarise the following rules in a concise and insightful manner:\n'
    + '\n'.join(coin_rules)
)
formatted_summary = ".\n    ".join(coin_context.split('.'))
print(f"Summary: {formatted_summary}")

Summary: In a game with two players, Red and Blue, on a grid, one coin is randomly placed each round.
     Players move simultaneously, aiming to reach the coin for points.
     If they pick up the opposite color, the other player loses points.
     The goal is to maximize both players' points.
     The grid is a 2D array with "R", "B", and "C" for Red, Blue, and coin.
     Valid moves are "left", "right", "up", or "down".
    


In [5]:
# @title Setup Clock

INIT_TIME = datetime(year = 2024, month = 7, day = 3, hour = 10)

clock = FixedIntervalClock(
    start = INIT_TIME,
    step_size = timedelta(seconds = 1)
)

In [6]:
# @title Setup Memory

player_importance = ConstantImportanceModel()
gm_importance = ConstantImportanceModel()

blank_factory = MemoryFactory(
    model = model,
    embedder = embedder,
    importance = player_importance.importance,
    clock_now = clock.now,
)

formative_factory = FormativeMemoryFactory(
    model = model,
    shared_memories = coin_rules,
    blank_memory_factory_call = blank_factory.make_blank_memory
)

In [7]:
# @title Agent Building

"""Builds an agent to play the Coin game with the given configuration.

Args:
    config (AgentConfig): The configuration of the agent.

Returns:
    (BasicAgent, AssociativeMemory): The agent and its memory object.
"""
def build_agent(config: AgentConfig) -> tuple[BasicAgent, AssociativeMemory]:
    memory = formative_factory.make_memories(config)

    timeComp = agent_components.report_function.ReportFunction(
        name = "Current Time",
        function = clock.current_time_interval_str,
    )

    idComp = agent_components.identity.SimIdentity(model, memory, config.name)
    planComp = agent_components.plan.SimPlan(
        model = model, memory = memory, agent_name = config.name,
        clock_now = clock.now, components = [idComp], verbose = False,
        goal = agent_components.constant.ConstantComponent(state = config.goal),
    )

    observeComp = agent_components.observation.Observation(
        agent_name = config.name, memory = memory, clock_now = clock.now,
        timeframe = clock.get_step_size(), component_name = "Observation",
    )
    summaryComp = agent_components.observation.ObservationSummary(
        agent_name = config.name, model = model, clock_now = clock.now,
        memory = memory, components = [idComp], component_name = "Summary",
        timeframe_delta_from = timedelta(minutes = 10),
        timeframe_delta_until = timedelta(minutes = 0),
    )

    agent = BasicAgent(
        agent_name = config.name, model = model, clock = clock, verbose = True,
        components = [idComp, planComp, timeComp, observeComp, summaryComp]
    )

    return agent, memory

# TODO: Add "interesting" traits to experiment.
player_configs =  map(
    lambda name: AgentConfig(
        name = name,
        goal = "Maximise every player's points.",
        context = coin_context
    ),
    ["Red", "Blue"]
)

players = {}
with ThreadPoolExecutor(max_workers = 2) as executor:
    for agent, memory in executor.map(build_agent, player_configs):
        players[agent.name] = {'player': agent, 'memory': memory}


Number of generated formative episodes (8) does not match number of formative ages (7).


In [8]:
# @title Game Master Building

gm_memory = AssociativeMemory(
    sentence_embedder = embedder,
    importance = gm_importance.importance,
    clock = clock.now,
)

rulesComp = agent_components.constant.ConstantComponent(
    state = '\n'.join(coin_rules), name = "Coin Game Rules"
)

coinComp = CoinState(
    clock_now = clock.now,
    model = model,
    red_memory = players["Red"]['memory'],
    blue_memory = players["Blue"]['memory'],
    width = COIN_GRID_WIDTH,
    height = COIN_GRID_HEIGHT,
    verbose = True,
)

env = GameMaster(
    model = model,
    memory = gm_memory,
    clock = clock,
    components = [rulesComp, coinComp],
    players = list(map(lambda player: player['player'], players.values())),
    verbose = True,
)

In [9]:
for _ in range(2):
  env.step()

# @title Summarize the entire story
all_gm_memories = env._memory.retrieve_recent(k=10000, add_time=True)

detailed_story = '\n'.join(all_gm_memories)
print('len(detailed_story): ', len(detailed_story))
# print(detailed_story)

episode_summary = model.sample_text(
    f'Sequence of events:\n{detailed_story}'+
    '\nNarratively summarize the above temporally ordered ' +
    'sequence of events. Write it as a news report. Summary:\n',
     max_tokens=3500, terminators=())
print(episode_summary)


Red context of action:
Red's Current Time:
 03 Jul 2024 [10:00:00 - 10:00:01]

Red's Observation:
[03 Jul 2024 10:00:00] [observation] Red: 0, Blue: 0, Grid: 2x2
[03 Jul 2024 10:00:00] [observation] BR
[03 Jul 2024 10:00:00] [observation] .C


Question: What would Red do for the next 1 second? Give a specific activity. Pick an activity that would normally take about 1 second to complete. If the selected action has a direct or indirect object then it must be specified explicitly. For example, it is valid to respond with "Red votes for Caroline because..." but not valid to respond with "Red votes because...".
Answer: Red will quickly scan the grid for any anomalies.


GM context of action and chain of thought:
Instructions: This is a social science experiment. It is structured as a tabletop roleplaying game (like dungeons and dragons). You are the game master. You will describe the current situation to the participants in the experiment and then on the basis of what you tell them they w